In [ ]:
import matplotlib as mpl

mpl.rcParams.update({

    # -------- Lines --------
    "lines.linewidth": 2,
    "lines.markersize": 4.5,

    # -------- Axes labels --------
    "axes.labelsize": 14,
    "axes.titlesize": 14,

    # -------- Tick labels --------
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    # -------- Legend --------
    "legend.fontsize": 11,

    # -------- Grid --------
    "grid.alpha": 0.25,

})

plot A function

In [ ]:
from pathlib import Path

def plot_A(ax, file_path: Path | str = None):
    """
    RNA L=18 — Plot A (self-contained)
    Fixed bin width = 1.0 with adaptive log-inset floor.
    """
    import numpy as np
    import pandas as pd
    from pathlib import Path
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    default_path = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/RNA/L_18/"
        "plot_a/plot_a_files/Plot_A_RNA_L18_Billion_Samples.csv"
    )
    file_path = Path(file_path) if file_path is not None else default_path

    if not file_path.exists():
        raise FileNotFoundError(f"CSV not found: {file_path}")

    print(f"\nLoading from: {file_path}")
    df = pd.read_csv(file_path, sep=r"\s+|\t|,", engine="python")
    df.columns = df.columns.str.strip()

    if "Status" not in df.columns or "Freq" not in df.columns or "Complexity" not in df.columns:
        raise ValueError(f"CSV missing expected columns (Status, Freq, Complexity): {file_path}")

    df = df[df["Status"].isin(["Rand", "0muts"])]
    df_rand = df[df["Status"] == "Rand"].copy()
    df_p = df[df["Status"] == "0muts"].copy()

    df_rand["RelFreq"] = df_rand["Freq"] / df_rand["Freq"].sum()
    df_p["RelFreq"] = df_p["Freq"] / df_p["Freq"].sum()

    n_rand = len(df_rand)
    n_p = len(df_p)

    all_vals = np.concatenate([
        df_rand["Complexity"].to_numpy(),
        df_p["Complexity"].to_numpy()
    ])

    if all_vals.size == 0:
        raise RuntimeError("No complexity values found in CSV.")

    min_val = float(np.min(all_vals))
    max_val = float(np.max(all_vals))

    bin_width = 1.0
    bins = np.arange(min_val, max_val + bin_width, bin_width)
    n_bins = len(bins) - 1

    rand_counts, bin_edges = np.histogram(
        df_rand["Complexity"], bins=bins, weights=df_rand["RelFreq"]
    )
    p_counts, _ = np.histogram(
        df_p["Complexity"], bins=bins, weights=df_p["RelFreq"]
    )

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    nonzero = np.concatenate([
        rand_counts[rand_counts > 0],
        p_counts[p_counts > 0]
    ])

    eps = float(np.min(nonzero) * 0.1) if nonzero.size > 0 else 1e-12

    rand_safe = np.clip(rand_counts, eps, None)
    p_safe = np.clip(p_counts, eps, None)

    g_dist_complexity_mean = (
        (df_rand["Complexity"] * df_rand["Freq"]).sum()
        / df_rand["Freq"].sum()
    )

    p_dist_complexity_mean = (
        (df_p["Complexity"] * df_p["Freq"]).sum()
        / df_p["Freq"].sum()
    )
    print(f"G-dist mean complexity: {g_dist_complexity_mean:.6f}")
    print(f"P-dist mean complexity: {p_dist_complexity_mean:.6f}")
    print(f"Number of samples in G-dist: {n_rand}")
    print(f"Number of samples in P-dist: {n_p}")


    # -------------------------
    # Main plot
    # -------------------------
    ax.plot(bin_centers, rand_counts,
            marker="o",
            color="orange",
            label="G-dist")

    ax.fill_between(bin_centers, rand_counts, 0,
                    color="orange",
                    alpha=0.18)

    ax.plot(bin_centers, p_counts,
            marker="o",
            color="blue",
            label="P-dist")

    ax.fill_between(bin_centers, p_counts, 0,
                    color="blue",
                    alpha=0.15)

    ax.set_xlabel("Complexity (Lempel-Ziv)")
    ax.set_ylabel("Relative Frequency")

    ax.legend(frameon=False, loc="upper left")

    #ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.set_axisbelow(True)

    ax.set_ylim(0, 0.7)

    # -------------------------
    # Inset
    # -------------------------
    ax_inset = inset_axes(ax, width="22%", height="22%", loc="upper right")

    bin_spacing = bin_centers[1] - bin_centers[0] if len(bin_centers) > 1 else 0.0
    offset = 0.10 * bin_spacing

    ax_inset.plot(bin_centers - offset, rand_safe, color="orange")
    ax_inset.plot(bin_centers + offset, p_safe, color="blue")

    ax_inset.set_yscale("log")
    ax_inset.set_xlabel("Complexity", fontsize=9)
    ax_inset.set_ylabel("Log Frequency", fontsize=9)
    ax_inset.tick_params(labelsize=8)

    return {
        "bin_width": bin_width,
        "n_bins": n_bins,
        "epsilon": eps,
        "n_rand": int(n_rand),
        "n_p": int(n_p),
    }

plot B function

In [ ]:
def plot_B_RNA18(ax):
    """
    RNA L=18 — Plot B
    Entropy and global scaled complexity vs sample size.
    """

    import pandas as pd
    from pathlib import Path

    p18 = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/RNA/L_18/"
        "plot_b/plot_b_files/RNA_L18_global_local_sampling.csv"
    )

    df18 = pd.read_csv(p18)

    rename_map = {
        "Samples": "sample_size",
        "Entropy": "entropy",
        "GlobalScaled": "scaled_global"
    }

    df18 = df18.rename(columns=rename_map)

    df18["sample_size"] = pd.to_numeric(df18["sample_size"], errors="coerce")
    df18["entropy"] = pd.to_numeric(df18["entropy"], errors="coerce")
    df18["scaled_global"] = pd.to_numeric(df18["scaled_global"], errors="coerce")

    df18.sort_values("sample_size", inplace=True)

    N = df18["sample_size"].values
    S = df18["entropy"].values
    Kg = df18["scaled_global"].values

    # -------------------------
    # Plot
    # -------------------------
    ax.semilogx(N, S, "o-",
                label=r"$S$",
                color="green")

    ax.semilogx(N, Kg, "o-",
                label=r"$\langle \tilde{K}_{g} \rangle$",
                color="orange")

    ax.set_xlabel("Number of sampled genotypes (N, log scale)")
    ax.set_ylabel(r'Entropy (S) and scaled $\langle \tilde{K}_{g} \rangle$')
    ax.set_xscale("log")
    ax.set_xlim(5, 2_000_000_000)
    ax.minorticks_off()
    ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.margins(x=0.05, y=0.05)

    ax.legend(frameon=False)

NO plot C function! This is the appendix figure

plot D function

In [ ]:
def plot_D(ax):
    """
    RNA L=18 — Plot D
    Mean G1..G5 vs mutation number.
    """

    import numpy as np
    import pandas as pd
    from pathlib import Path

    csv_path = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/circadian/"
        "mut_project_updates/figures/RNA/L_18/"
        "plot_d/plot_d_files/RNA_L18_k_scaled_1_for_plot_D.csv"
    )

    marker = "o"

    # --------------------------------
    # Load CSV
    # --------------------------------
    df = pd.read_csv(csv_path)
    if df.empty:
        raise RuntimeError(f"No data found in {csv_path}")

    first_col = df.columns[0]
    first_col_lower = first_col.lower()

    if any(k in first_col_lower for k in ("mut", "number", "n", "k")):
        x_vals = df.iloc[:, 0].to_numpy()
        data_df = df.iloc[:, 1:].copy()
    else:
        x_vals = np.arange(len(df))
        data_df = df.copy()

    # Find MeanG* columns
    meang_cols = [
        c for c in data_df.columns
        if str(c).lower().startswith("meang")
    ]

    if not meang_cols:
        meang_cols = [
            c for c in data_df.columns
            if any(s in str(c).lower() for s in ("g1","g2","g3","g4","g5"))
        ]

    if not meang_cols:
        raise RuntimeError("No MeanG* columns found in CSV.")

    # Prepare arrays
    series_vals = []
    for c in meang_cols:
        vals = pd.to_numeric(data_df[c], errors="coerce").to_numpy()
        series_vals.append(vals)

    series_vals = np.array(series_vals)

    # x positions
    x_positions = np.arange(len(x_vals))

    if np.issubdtype(x_vals.dtype, np.number):
        x_tick_labels = [
            str(int(x)) if float(x).is_integer() else f"{x:.3g}"
            for x in x_vals
        ]
    else:
        x_tick_labels = [str(x) for x in x_vals]

    # --------------------------------
    # Plot (pink → plum gradient G1 → G5)
    # --------------------------------
    palette = [
        "#E8B5D6",  # G1
        "#CC79A7",  # G2
        "#A63478",  # G3
        "#7A1F5C",  # G4
        "#4A0D3A",  # G5
    ]

    for i, vals in enumerate(series_vals):
        label = f"G{i+1}"
        ax.plot(x_positions, vals,
                marker=marker,
                color=palette[i],
                label=label)

    ax.set_xlabel("Number of mutations")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")

    # show every other tick
    ax.set_xticks(x_positions[::2])
    ax.set_xticklabels(x_tick_labels[::2])

    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    # Reverse legend (G5 → G1)
    handles, legend_labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], legend_labels[::-1],
              loc="upper right",
              frameon=False)

    # 0.5 tick spacing margin
    if len(x_positions) > 1:
        spacing = x_positions[1] - x_positions[0]
    else:
        spacing = 1.0

    ax.set_xlim(x_positions[0] - 0.5 * spacing,
                x_positions[-1] + 0.5 * spacing)

create master figure

In [ ]:
import matplotlib.pyplot as plt

# Create 2x2 layout
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Remove bottom-right axis (we will re-center bottom-left)
fig.delaxes(axes[1, 1])

# Call plotting functions
plot_A(axes[0, 0])
plot_B_RNA18(axes[0, 1])
plot_D(axes[1, 0])

# Let matplotlib compute layout first
plt.tight_layout()
plt.subplots_adjust(hspace=0.28, wspace=0.25)

# --- Re-center bottom-left axis ---
top_left_pos = axes[0, 0].get_position()
top_right_pos = axes[0, 1].get_position()
bottom_pos = axes[1, 0].get_position()

center_left = top_left_pos.x0 + 0.5 * top_left_pos.width
center_right = top_right_pos.x0 + 0.5 * top_right_pos.width
target_center = 0.5 * (center_left + center_right)

new_x0 = target_center - 0.5 * bottom_pos.width

axes[1, 0].set_position([new_x0, bottom_pos.y0,
                         bottom_pos.width, bottom_pos.height])

# Panel labels (standardized)
axes[0, 0].text(0.5, -0.14, "(a)",
                transform=axes[0, 0].transAxes,
                ha='center', va='top', fontsize=11)

axes[0, 1].text(0.5, -0.14, "(b)",
                transform=axes[0, 1].transAxes,
                ha='center', va='top', fontsize=11)

axes[1, 0].text(0.5, -0.14, "(c)",
                transform=axes[1, 0].transAxes,
                ha='center', va='top', fontsize=11)

plt.show()